# 05. Department & Aisle Performance
**Enterprise Retail Intelligence & Decision Engine**  
*Phase 8: Python Exploratory Data Analysis & Business Intelligence*

---

### Overview & Objectives
Department market share, reorder rates, and category contribution.

---


## 1. Setup & Department Joining

Join train order items with products, departments, and aisles.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path("../data/Processed")
df_train = pd.read_parquet(DATA_DIR / "order_products_train_clean.parquet")
df_products = pd.read_parquet(DATA_DIR / "products_clean.parquet")
df_depts = pd.read_parquet(DATA_DIR / "departments_clean.parquet")
df_aisles = pd.read_parquet(DATA_DIR / "aisles_clean.parquet")

df_merged = df_train.merge(df_products, on='product_id').merge(df_depts, on='department_id')
print("Merged dataset ready.")

## 2. Department Volume & Market Share

Measure order volume and share across all 21 supermarket departments.

In [ ]:
dept_summary = df_merged.groupby('department').agg(
    total_items=('reordered', 'count'),
    reorders=('reordered', 'sum')
).reset_index()
dept_summary['reorder_rate'] = dept_summary['reorders'] / dept_summary['total_items']
dept_summary['share_pct'] = (dept_summary['total_items'] / dept_summary['total_items'].sum()) * 100
dept_summary = dept_summary.sort_values(by='total_items', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=dept_summary, x='share_pct', y='department', palette='rocket')
plt.title("Department Item Share (%) in Instacart Baskets")
plt.xlabel("Share of Total Items (%)")
plt.tight_layout()
plt.show()
dept_summary[['department', 'total_items', 'share_pct', 'reorder_rate']]

## 3. Department Reorder Rates

Identify departments driven by repeat subscriptions vs one-time purchases.

In [ ]:
dept_reorder_sorted = dept_summary.sort_values(by='reorder_rate', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=dept_reorder_sorted, x='reorder_rate', y='department', palette='mako')
plt.title("Reorder Rate by Department")
plt.xlabel("Reorder Rate")
plt.axvline(df_train['reordered'].mean(), color='red', linestyle='--', label=f"Average: {df_train['reordered'].mean():.2f}")
plt.legend()
plt.tight_layout()
plt.show()